# DLAI Model Merging - Pairwise merging pilot (seed 42)

This notebook merges all six pairs of task-specialized encoders and evaluates Mean, Task Arithmetic, and TIES. Each merged encoder is paired with the original task-specific head during evaluation.

Before running: use **Add Input** in Kaggle and attach the saved output of notebook 02 containing `pilot_specialists_seed42.zip`. Select GPU T4 x2 and enable Internet.

In [ ]:
!nvidia-smi
!python --version
!find /kaggle/input -maxdepth 3 -type f | head -50

## Install the latest project code and locate the specialist bundle

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

REPO = 'https://github.com/LeuxLello/Dlai-model-merging.git'
WORKDIR = Path('/kaggle/working/Dlai-model-merging')
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
subprocess.check_call(['git', 'clone', '--depth', '1', REPO, str(WORKDIR)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(WORKDIR)])
sys.path.insert(0, str(WORKDIR / 'src'))
os.chdir(WORKDIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

bundle_candidates = list(Path('/kaggle/input').rglob('pilot_specialists_seed42.zip'))
assert bundle_candidates, (
    'Specialist ZIP not found. Click Add Input and attach the saved output of notebook 02.'
)
bundle = bundle_candidates[0]
SPECIALISTS = Path('/kaggle/working/pilot_specialists_seed42')
if SPECIALISTS.exists():
    shutil.rmtree(SPECIALISTS)
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(SPECIALISTS)
print('Bundle:', bundle)
print('Encoder files:', list(SPECIALISTS.rglob('encoder.pt')))

## Environment and checkpoints

In [ ]:
import itertools, json, platform
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification
from dlai_merge.diagnostics import cosine_similarity, l2_norm, sign_agreement, subtract_states
from dlai_merge.evaluation import TaskEvaluator
from dlai_merge.merging import mean_merge, task_arithmetic, ties_merge

assert torch.cuda.is_available(), 'Enable GPU T4 x2.'
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print('GPU:', gpu_name, '| capability:', capability)
assert capability[0] >= 7, 'Use GPU T4 x2, not P100.'
torch.ones(1, device='cuda').add_(1)

TASKS = ['sst2', 'imdb', 'mrpc', 'rte']
SEED = 42
BASE_MODEL = 'prajjwal1/bert-mini'
def artifact(task, filename):
    matches = list(SPECIALISTS.rglob(f'{task}/seed-{SEED}/{filename}'))
    assert len(matches) == 1, (task, filename, matches)
    return matches[0]

encoders = {t: torch.load(artifact(t, 'encoder.pt'), map_location='cpu', weights_only=True) for t in TASKS}
heads = {t: torch.load(artifact(t, 'head.pt'), map_location='cpu', weights_only=True) for t in TASKS}
base_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
base_encoder = {k: v.detach().cpu().clone() for k, v in base_model.base_model.state_dict().items()}
vectors = {t: subtract_states(encoders[t], base_encoder) for t in TASKS}
print({t: round(l2_norm(vectors[t]), 4) for t in TASKS})

## Parameter-space diagnostics

In [ ]:
diagnostic_rows = []
for left, right in itertools.combinations(TASKS, 2):
    agreement = sign_agreement(vectors[left], vectors[right])
    diagnostic_rows.append({
        'task_a': left,
        'task_b': right,
        'pair': f'{left}+{right}',
        'cosine_similarity': cosine_similarity(vectors[left], vectors[right]),
        'sign_agreement': agreement,
        'sign_conflict': 1.0 - agreement,
        'norm_a': l2_norm(vectors[left]),
        'norm_b': l2_norm(vectors[right]),
    })
pair_diagnostics = pd.DataFrame(diagnostic_rows)
pair_diagnostics

## Re-evaluate specialists with a common evaluation path

In [ ]:
evaluators = {
    task: TaskEvaluator(task, heads[task], max_eval_samples=2000, seed=SEED, output_root='/kaggle/working/eval')
    for task in TASKS
}
specialist_rows = []
for task in TASKS:
    scores = evaluators[task].evaluate(encoders[task])
    specialist_rows.append({'task': task, 'primary_metric': evaluators[task].primary_metric, **scores})
specialist_scores = pd.DataFrame(specialist_rows)
specialist_reference = dict(zip(specialist_scores.task, specialist_scores.primary_score))
specialist_scores

## Merge and evaluate all pairs
For every pair: Mean once, Task Arithmetic at three scales, and TIES at three densities times three scales. Each merge is evaluated on both constituent tasks.

In [ ]:
result_rows = []
SCALES = [0.5, 0.75, 1.0]
DENSITIES = [0.1, 0.2, 0.5]

def record(pair, method, scale, density, merged):
    for task in pair:
        scores = evaluators[task].evaluate(merged)
        reference = specialist_reference[task]
        result_rows.append({
            'task_a': pair[0], 'task_b': pair[1], 'pair': '+'.join(pair),
            'method': method, 'scale': scale, 'density': density, 'eval_task': task,
            **scores,
            'specialist_score': reference,
            'retained_ratio': scores['primary_score'] / reference,
            'score_delta': scores['primary_score'] - reference,
        })

for pair in itertools.combinations(TASKS, 2):
    states = [encoders[pair[0]], encoders[pair[1]]]
    print('Merging', pair)
    record(pair, 'mean', 1.0, np.nan, mean_merge(base_encoder, states))
    for scale in SCALES:
        record(pair, 'task_arithmetic', scale, np.nan, task_arithmetic(base_encoder, states, scale))
    for density in DENSITIES:
        for scale in SCALES:
            record(pair, 'ties', scale, density, ties_merge(base_encoder, states, density, scale))

merge_results = pd.DataFrame(result_rows)
print('Evaluation rows:', len(merge_results))
merge_results.head()

## Aggregate both tasks and identify the best configuration per pair

In [ ]:
group_cols = ['pair', 'task_a', 'task_b', 'method', 'scale', 'density']
merge_summary = (merge_results.groupby(group_cols, dropna=False)
    .agg(mean_retained=('retained_ratio', 'mean'), worst_retained=('retained_ratio', 'min'),
         mean_delta=('score_delta', 'mean'))
    .reset_index())
best_per_pair = (merge_summary.sort_values(['pair', 'mean_retained'], ascending=[True, False])
    .groupby('pair', as_index=False).head(1))
best_per_pair

## Pilot figures

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

FIGURES = Path('/kaggle/working/merging_figures')
FIGURES.mkdir(exist_ok=True)
plot_data = best_per_pair.merge(pair_diagnostics[['pair', 'cosine_similarity', 'sign_conflict']], on='pair')
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.scatterplot(data=plot_data, x='cosine_similarity', y='mean_retained', hue='pair', s=100, ax=axes[0])
sns.scatterplot(data=plot_data, x='sign_conflict', y='mean_retained', hue='pair', s=100, ax=axes[1], legend=False)
axes[0].axhline(1.0, color='gray', linestyle='--'); axes[1].axhline(1.0, color='gray', linestyle='--')
axes[0].set_title('Task-vector alignment'); axes[1].set_title('Sign conflict')
fig.tight_layout(); fig.savefig(FIGURES / 'alignment_vs_retention.png', dpi=180, bbox_inches='tight')
plt.show()

## Export compact results

In [ ]:
OUT = Path('/kaggle/working/merging_pilot_results')
OUT.mkdir(exist_ok=True)
pair_diagnostics.to_csv(OUT / 'pair_diagnostics.csv', index=False)
specialist_scores.to_csv(OUT / 'specialist_recheck.csv', index=False)
merge_results.to_csv(OUT / 'merge_results.csv', index=False)
merge_summary.to_csv(OUT / 'merge_summary.csv', index=False)
best_per_pair.to_csv(OUT / 'best_per_pair.csv', index=False)
shutil.copy2(FIGURES / 'alignment_vs_retention.png', OUT / 'alignment_vs_retention.png')
metadata = {
    'purpose': 'pairwise merging pilot; seed 42; hyperparameters not final',
    'seed': SEED, 'tasks': TASKS, 'scales': SCALES, 'densities': DENSITIES,
    'gpu': gpu_name, 'python': platform.python_version(), 'torch': torch.__version__,
}
(OUT / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
archive = shutil.make_archive('/kaggle/working/merging_pilot_seed42', 'zip', OUT)
print(archive)
print(*sorted(str(p) for p in OUT.iterdir()), sep='\n')